# LH Nautical Data Challenge - Pipeline Completo

Este Notebook contém toda a arquitetura de dados (EDA, Modelagem, Análises, Previsão de Demanda e Relatórios) consolidada para execução no Google Colab.

**Pré-requisito:**
Você deve fazer o upload da pasta `1-lh_nautical_csv` (ou seu conteúdo .zip) para o ambiente do Colab antes de rodar.

In [ ]:
# Preparação do Ambiente no Colab
!pip install -q statsmodels pandas plotly pytest

import os
BASE_DIR = '.'
os.makedirs(os.path.join(BASE_DIR, 'processed'), exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, 'outputs', 'reports'), exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, 'outputs', 'figures'), exist_ok=True)

# Caso tenha subido o ZIP com os CSVs brutos, descomente e rode a linha abaixo:
# !unzip -q -o 1-lh_nautical_csv.zip -d 1-lh_nautical_csv


## Etapa: `01_eda.py`

In [ ]:
"""
=======================================================================
SCRIPT 01 - EDA (Exploratory Data Analysis) - LH Nautical
=======================================================================
Responsavel : Gabriel Santos (Tech Lead) - Etapa de entendimento inicial
Objetivo    : Gerar um relatorio detalhado de qualidade dos dados brutos.
              Responde a pergunta: "Do que precisamos limpar antes de usar?"
=======================================================================
"""

import os
import sys
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Sem interface grafica
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

# ---- Configuracao de caminhos (relativo ao script) ----
BASE_DIR    = Path(__file__).parent
CSV_DIR     = BASE_DIR / "1-lh_nautical_csv"
FIG_DIR     = BASE_DIR / "outputs" / "figures"
REPORT_DIR  = BASE_DIR / "outputs" / "reports"

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

# ---- Estilo de visualizacao ----
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d2e',
    'axes.edgecolor':   '#3a3d5c',
    'axes.labelcolor':  '#c8cce0',
    'xtick.color':      '#c8cce0',
    'ytick.color':      '#c8cce0',
    'text.color':       '#ffffff',
    'grid.color':       '#2a2d3e',
    'grid.alpha':       0.5,
    'font.family':      'sans-serif',
})

print("=" * 60)
print("  LH Nautical - Script 01: EDA & Qualidade dos Dados")
print("=" * 60)

# ---- 1. Carregamento de todos os CSVs ----
print("\n[1/4] Carregando arquivos CSV...")
csv_files = [f for f in os.listdir(CSV_DIR) if f.endswith('.csv')]
dataframes = {}
summary_rows = []

for fname in sorted(csv_files):
    path = os.path.join(CSV_DIR, fname)
    try:
        df = pd.read_csv(path)
        dataframes[fname] = df
        total_cells = df.shape[0] * df.shape[1]
        missing_cells = df.isnull().sum().sum()
        missing_pct = (missing_cells / total_cells * 100) if total_cells > 0 else 0
        dup_rows = df.duplicated().sum()
        summary_rows.append({
            "Arquivo":             fname.replace('.csv', ''),
            "Linhas":              df.shape[0],
            "Colunas":             df.shape[1],
            "Celulas_Nulas":       int(missing_cells),
            "Pct_Nula":            round(missing_pct, 2),
            "Linhas_Duplicadas":   int(dup_rows),
        })
        print(f"  [OK] {fname}: {df.shape[0]:>7,} linhas | {df.shape[1]:>2} colunas | {missing_pct:.1f}% nulo")
    except Exception as e:
        print(f"  [ERRO] {fname}: {e}")

# ---- 2. Relatorio de Qualidade ----
print("\n[2/4] Gerando relatorio de qualidade...")
summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values('Pct_Nula', ascending=False)
report_path = os.path.join(REPORT_DIR, "01_qualidade_dados.csv")
summary_df.to_csv(report_path, index=False, encoding='utf-8-sig')
print(f"  Relatorio salvo em: {report_path}")

# ---- 3. Grafico: % de Nulos por Tabela ----
print("\n[3/4] Gerando grafico de dados nulos...")
fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor('#0f1117')

cores = ['#e05a6a' if x > 10 else '#5a9fd4' for x in summary_df['Pct_Nula']]
bars = ax.barh(summary_df['Arquivo'], summary_df['Pct_Nula'], color=cores, edgecolor='none')

for bar, val in zip(bars, summary_df['Pct_Nula']):
    if val > 0:
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
                f'{val:.1f}%', va='center', fontsize=8, color='#c8cce0')

ax.set_xlabel('% de celulas nulas', fontsize=10)
ax.set_title('Qualidade dos Dados - % de Nulos por Tabela', fontsize=13, fontweight='bold', pad=15)
ax.axvline(x=10, color='#e05a6a', linestyle='--', alpha=0.5, label='Limite 10%')
ax.legend(fontsize=8)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
fig_path = os.path.join(FIG_DIR, "01_nulos_por_tabela.png")
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"  Figura salva em: {fig_path}")

# ---- 4. Grafico: Volume de Linhas por Tabela ----
print("\n[4/4] Gerando grafico de volume de dados...")
sorted_vol = summary_df.sort_values('Linhas', ascending=True)
fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor('#0f1117')

bars = ax.barh(sorted_vol['Arquivo'], sorted_vol['Linhas'],
               color='#7b5ea7', edgecolor='none')

for bar, val in zip(bars, sorted_vol['Linhas']):
    ax.text(bar.get_width() + 200, bar.get_y() + bar.get_height() / 2,
            f'{val:,.0f}', va='center', fontsize=8, color='#c8cce0')

ax.set_xlabel('Numero de linhas', fontsize=10)
ax.set_title('Volume de Dados por Tabela', fontsize=13, fontweight='bold', pad=15)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
fig_path = os.path.join(FIG_DIR, "01_volume_por_tabela.png")
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"  Figura salva em: {fig_path}")

print("\n[CONCLUIDO] Script 01 - EDA finalizado com sucesso.")
print(f"  Relatorios em: {REPORT_DIR}")
print(f"  Figuras em:    {FIG_DIR}")


## Etapa: `02_tratamento_modelagem.py`

In [ ]:
"""
=======================================================================
SCRIPT 02 - Tratamento e Modelagem de Dados - LH Nautical
=======================================================================
Responsavel : Gabriel Santos (Tech Lead) - Etapa de Engenharia de Dados
Objetivo    : Limpar, tipar e modelar os dados brutos em tabelas
              prontas para analise (Fato de Vendas + Dimensoes).

              Premissas:
              - Datas convertidas para datetime
              - Valores monetarios como float
              - Nulos tratados com criterio (remover ou imputar)
              - Duplicatas removidas
              - Dados exportados para /processed/
=======================================================================
"""

import os
import gc
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

try:
    import duckdb
    _DUCKDB_AVAILABLE = True
except ImportError:
    _DUCKDB_AVAILABLE = False

try:
    import psutil
    _PSUTIL_AVAILABLE = True
except ImportError:
    _PSUTIL_AVAILABLE = False

# ---- Configuracao (relativo ao script) ----
BASE_DIR    = Path(__file__).parent
CSV_DIR     = BASE_DIR / "1-lh_nautical_csv"
PROC_DIR    = BASE_DIR / "processed"
REPORT_DIR  = BASE_DIR / "outputs" / "reports"
DB_PATH     = BASE_DIR / "nautical.duckdb"

os.makedirs(PROC_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

# ---- Conexao DuckDB (opcional) ----
_db = None
if _DUCKDB_AVAILABLE:
    _db = duckdb.connect(str(DB_PATH))
    print(f"  [DuckDB] Banco conectado: {DB_PATH}")
else:
    print("  [DuckDB] duckdb nao instalado — escrevendo apenas CSVs.")
    print("           Para ativar: pip install duckdb")

def _ram_mb() -> str:
    """Retorna uso atual de RAM em MB (requer psutil)."""
    if _PSUTIL_AVAILABLE:
        proc = psutil.Process(os.getpid())
        return f"{proc.memory_info().rss / 1_048_576:.0f} MB"
    return "N/A"

def read_csv(name: str, dtypes: dict | None = None) -> pd.DataFrame:
    """Leitura padrao de CSV com dtypes opcionais para economia de RAM."""
    return pd.read_csv(os.path.join(CSV_DIR, name), dtype=dtypes)

def save(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Salva DataFrame em /processed/ (CSV) e, se disponivel, no DuckDB."""
    path = os.path.join(PROC_DIR, name)
    df.to_csv(path, index=False, encoding='utf-8-sig')
    # --- Escrita no DuckDB (adicional, nao substitui o CSV) ---
    if _db is not None:
        table_name = name.replace('.csv', '')
        _db.execute(f"DROP TABLE IF EXISTS {table_name}")
        _db.execute(f"CREATE TABLE {table_name} AS SELECT * FROM df")
    print(f"  [SALVO] {name}: {len(df):,} linhas  |  RAM: {_ram_mb()}")
    return df

print("=" * 60)
print("  LH Nautical - Script 02: Tratamento e Modelagem")
print("=" * 60)

# ====================================================================
# DIMENSAO: CLIENTES (dim_customers)
# ====================================================================
print("\n[1/8] Tratando dim_customers...")
customers = read_csv('customers.csv')
customers = customers.drop_duplicates(subset=['id'])
customers['created_at'] = pd.to_datetime(customers['created_at'], errors='coerce')
customers['updated_at'] = pd.to_datetime(customers['updated_at'], errors='coerce')
# Preenche trade_name ausente com legal_name (PF nao tem trade_name)
customers['trade_name'] = customers['trade_name'].fillna(customers['legal_name'])
# Mascara email ausente como 'sem_email@lhnautical.com'
customers['email'] = customers['email'].fillna('sem_email@lhnautical.com')
customers['phone'] = customers['phone'].fillna('N/A')
customers['state_registration'] = customers['state_registration'].fillna('NAO_INFORMADO')
dim_customers = customers.rename(columns={'id': 'customer_id'})
save(dim_customers, 'dim_customers.csv')

# ====================================================================
# DIMENSAO: PRODUTOS (dim_products)
# ====================================================================
print("\n[2/8] Tratando dim_products...")
products  = read_csv('products.csv')
brands    = read_csv('brands.csv')[['id', 'name']].rename(columns={'id': 'brand_id', 'name': 'brand_name'})
categories = read_csv('categories.csv')[['id', 'name']].rename(columns={'id': 'category_id', 'name': 'category_name'})
variants  = read_csv('product_variants.csv')

products = products.drop_duplicates(subset=['id'])
products['description'] = products['description'].fillna('Sem descricao')
products = products.merge(brands, on='brand_id', how='left')
products = products.merge(categories, on='category_id', how='left')
products['brand_name'] = products['brand_name'].fillna('Marca Desconhecida')
products['category_name'] = products['category_name'].fillna('Categoria Desconhecida')

# Junta variantes (SKU, custo, preco) com produtos
variants = variants.drop_duplicates(subset=['id'])
variants_slim = variants[['id', 'product_id', 'sku', 'cost_price', 'sale_price', 'weight_kg', 'is_active']].rename(
    columns={'id': 'variant_id', 'sale_price': 'selling_price'}
)
dim_products = variants_slim.merge(
    products[['id', 'name', 'brand_name', 'category_name', 'unit_of_measure', 'is_active']].rename(
        columns={'id': 'product_id', 'name': 'product_name', 'is_active': 'product_active'}
    ),
    on='product_id', how='left'
)
save(dim_products, 'dim_products.csv')

# ====================================================================
# DIMENSAO: LOCAIS (dim_locations)
# ====================================================================
print("\n[3/8] Tratando dim_locations...")
locations = read_csv('locations.csv')
locations = locations.drop_duplicates(subset=['id'])
locations['complement'] = locations['complement'].fillna('')
dim_locations = locations.rename(columns={'id': 'location_id'})
save(dim_locations, 'dim_locations.csv')

# ====================================================================
# DIMENSAO: TEMPO (dim_calendar) - Preenchida a partir das datas de pedidos
# ====================================================================
print("\n[4/8] Construindo dim_calendar...")
orders = read_csv('orders.csv')
orders['placed_at'] = pd.to_datetime(orders['placed_at'], errors='coerce')
date_range = pd.date_range(start='2020-01-01', end='2026-12-31', freq='D')
dim_calendar = pd.DataFrame({'date': date_range})
dim_calendar['year']        = dim_calendar['date'].dt.year
dim_calendar['quarter']     = dim_calendar['date'].dt.quarter
dim_calendar['month']       = dim_calendar['date'].dt.month
dim_calendar['month_name']  = dim_calendar['date'].dt.strftime('%B')
dim_calendar['week']        = dim_calendar['date'].dt.isocalendar().week.astype(int)
dim_calendar['day_of_week'] = dim_calendar['date'].dt.dayofweek         # 0=Segunda
dim_calendar['day_name']    = dim_calendar['date'].dt.strftime('%A')
dim_calendar['is_weekend']  = dim_calendar['day_of_week'].isin([5, 6])
save(dim_calendar, 'dim_calendar.csv')

# ====================================================================
# FATO: VENDAS (fato_vendas)  — OTIMIZADO PARA MEMORIA
# ====================================================================
print("\n[5/8] Construindo fato_vendas...")
print(f"  [RAM] Antes do join: {_ram_mb()}")

# Dtypes otimizados: colunas de alta cardinalidade como category,
# IDs como int32, monetarios como float32  -->  ~40-60% menos RAM
_orders_dtypes = {
    'id':              'int32',
    'customer_id':     'int32',
    'location_id':     'Int32',   # nullable int (pode ter nulos)
    'salesperson_id':  'Int32',
    'channel':         'category',
    'status':          'category',
}
_items_dtypes = {
    'id':                 'int32',
    'order_id':           'int32',
    'product_variant_id': 'Int32',
    'quantity':           'int32',
    'unit_price':         'float32',
    'line_total':         'float32',
}

order_items = read_csv('order_items.csv', dtypes=_items_dtypes)
orders['placed_at']   = pd.to_datetime(orders['placed_at'],   errors='coerce')
orders['created_at']  = pd.to_datetime(orders['created_at'],  errors='coerce')
orders['updated_at']  = pd.to_datetime(orders['updated_at'],  errors='coerce')

# Remove pedidos sem data valida
orders = orders.dropna(subset=['placed_at'])
orders = orders.drop_duplicates(subset=['id'])
order_items = order_items.drop_duplicates(subset=['id'])

# Junta itens com pedidos (apenas as colunas necessarias para reduzir footprint)
_orders_slim = orders[[
    'id', 'order_number', 'channel', 'customer_id', 'salesperson_id',
    'location_id', 'status', 'discount_amount', 'total', 'placed_at'
]].copy()

fato_vendas = order_items.merge(
    _orders_slim,
    left_on='order_id', right_on='id', how='inner', suffixes=('', '_order')
)
fato_vendas = fato_vendas.rename(columns={'id': 'order_item_id', 'order_id': 'order_id'})

# Libera memoria dos DataFrames intermediarios que nao serao mais usados
del _orders_slim, order_items
gc.collect()
print(f"  [RAM] Apos merge ordens+itens: {_ram_mb()}")

# Junta custo do produto (variante)
cost_data = dim_products[['variant_id', 'cost_price', 'product_name', 'brand_name', 'category_name', 'product_id']].copy()
fato_vendas = fato_vendas.merge(cost_data, left_on='product_variant_id', right_on='variant_id', how='left')

# Calculo de margem
fato_vendas['cost_total']   = (fato_vendas['cost_price'] * fato_vendas['quantity']).astype('float32')
fato_vendas['gross_profit'] = (fato_vendas['line_total'] - fato_vendas['cost_total']).astype('float32')
fato_vendas['margin_pct']   = (fato_vendas['gross_profit'] / fato_vendas['line_total'] * 100).round(2)

# Data como coluna Date para cruzamento com dim_calendar
fato_vendas['order_date']   = fato_vendas['placed_at'].dt.normalize()
fato_vendas['year']         = fato_vendas['placed_at'].dt.year.astype('int16')
fato_vendas['month']        = fato_vendas['placed_at'].dt.month.astype('int8')
fato_vendas['day_of_week']  = fato_vendas['placed_at'].dt.dayofweek.astype('int8')

print(f"  [RAM] Apos join de custo: {_ram_mb()}")
save(fato_vendas, 'fato_vendas.csv')

# ====================================================================
# FATO: PAGAMENTOS (fato_payments)
# ====================================================================
print("\n[6/8] Tratando fato_payments...")
payments = read_csv('payments.csv')
payments = payments.drop_duplicates(subset=['id'])
payments['paid_at']    = pd.to_datetime(payments['paid_at'],    errors='coerce')
payments['created_at'] = pd.to_datetime(payments['created_at'], errors='coerce')
fato_payments = payments.merge(
    orders[['id', 'customer_id', 'placed_at', 'channel', 'location_id']],
    left_on='order_id', right_on='id', how='left', suffixes=('', '_order')
)
save(fato_payments, 'fato_payments.csv')

# ====================================================================
# FATO: DEVOLUCOES (fato_returns)
# ====================================================================
print("[7/8] Tratando fato_returns...")
returns      = read_csv('returns.csv')
return_items = read_csv('return_items.csv')
returns = returns.drop_duplicates(subset=['id'])
returns['created_at'] = pd.to_datetime(returns['created_at'], errors='coerce')
returns['updated_at'] = pd.to_datetime(returns['updated_at'], errors='coerce')

# return_items nao tem product_variant_id direto; faz join via order_items
order_items_slim = read_csv('order_items.csv')[['id', 'product_variant_id']].rename(columns={'id': 'order_item_id'})
return_items = return_items.merge(order_items_slim, on='order_item_id', how='left')

fato_returns = return_items.merge(
    returns[['id', 'order_id', 'customer_id', 'reason', 'status', 'total_refund_amount', 'created_at']],
    left_on='return_id', right_on='id', how='left'
)
fato_returns = fato_returns.merge(
    cost_data[['variant_id', 'cost_price', 'product_name']],
    left_on='product_variant_id', right_on='variant_id', how='left'
)
fato_returns['refund_value'] = fato_returns['quantity'] * fato_returns['unit_refund_amount']

# --- Normaliza motivos de devolucao para evitar fragmentacao na analise ---
def _normalizar_motivo(motivo):
    """Lower, strip, corrige typos frequentes e padroniza para texto canonical."""
    if not isinstance(motivo, str):
        return 'Outros'
    m = motivo.strip().lower()
    m = m.replace('  ', ' ')  # espaco duplo
    # Mapeamento de variantes conhecidas para canonical
    canonical = {
        'cliente desistiu da compra':        ['cliente desistiu da compra', 'cliente desistiu', 'cliente desistui da compra'],
        'compra duplicada':                  ['compra duplicada'],
        'produto avariado no transporte':    ['produto avariado no transporte', 'produt oavariado no transporte', 'produto variado no transporte'],
        'item nao corresponde a descricao':  ['item nao corresponde a descricao', 'item nao corresponde a descricao', 'item nao corressponde a descricao', 'item nao corresponde a descricaoo'],
        'tamanho/cor incorretos':            ['tamanho/cor incorretos', 'tamanho/cor iincorretos', 'tamanho/coor incorretos', 'tamanho/cor incorretoos', 'tamanho/cor iincorretos'],
        'produto com defeito de fabrica':    ['produto com defeito de fabrica', 'produto com  defeito de fabrica', 'produto com defeitoo de fabrica', 'produto ccom defeito de fabrica', 'produt ocom defeito de fabrica'],
    }
    # Remove acentos para comparacao robusta
    import unicodedata
    def sem_acento(s):
        return unicodedata.normalize('NFD', s).encode('ascii', 'ignore').decode()
    m_norm = sem_acento(m)
    for canonical_name, variantes in canonical.items():
        for v in variantes:
            if m_norm == sem_acento(v):
                return canonical_name
    return m if m else 'Outros'

fato_returns['reason'] = fato_returns['reason'].apply(_normalizar_motivo)

save(fato_returns, 'fato_returns.csv')

# ====================================================================
# FATO: ESTOQUE (fato_stock)
# ====================================================================
print("\n[8/8] Tratando fato_stock...")
stock_levels = read_csv('stock_levels.csv')
stock_mvmt   = read_csv('stock_movements.csv')
# stock_levels nao tem coluna 'id'; chave natural e product_variant_id + location_id
stock_levels = stock_levels.drop_duplicates(subset=['product_variant_id', 'location_id'])
stock_mvmt   = stock_mvmt.drop_duplicates(subset=['id'])
stock_mvmt['occurred_at'] = pd.to_datetime(stock_mvmt['occurred_at'], errors='coerce')
fato_stock = stock_mvmt.merge(
    cost_data[['variant_id', 'cost_price', 'product_name']],
    left_on='product_variant_id', right_on='variant_id', how='left'
)
# Salva niveis de estoque atual separado
stock_levels.to_csv(os.path.join(PROC_DIR, 'dim_stock_levels.csv'), index=False, encoding='utf-8-sig')
print(f"  [SALVO] dim_stock_levels.csv: {len(stock_levels):,} linhas")
save(fato_stock, 'fato_stock.csv')

# ====================================================================
# FASE 2: SUPPLY CHAIN, RH, FISCAL, LOGISTICA
# ====================================================================
print("\n[9/8] Tratando Phase 2 (RH, Fornecedores, Compras, Fiscal, Logistica)...")

# 1. dim_employees
employees = read_csv('employees.csv')
employees = employees.drop_duplicates(subset=['id'])
employees['hire_date']        = pd.to_datetime(employees['hire_date'],        errors='coerce')
employees['termination_date'] = pd.to_datetime(employees['termination_date'], errors='coerce')
dim_employees = employees.rename(columns={'id': 'employee_id'})
save(dim_employees, 'dim_employees.csv')

# 2. dim_suppliers
suppliers = read_csv('suppliers.csv')
suppliers = suppliers.drop_duplicates(subset=['id'])
dim_suppliers = suppliers.rename(columns={'id': 'supplier_id'})
save(dim_suppliers, 'dim_suppliers.csv')

# 3. dim_addresses
addresses = read_csv('addresses.csv')
addresses = addresses.drop_duplicates(subset=['id'])
dim_addresses = addresses.rename(columns={'id': 'address_id'})
save(dim_addresses, 'dim_addresses.csv')

# 4. fato_compras e goods_receipt_items
purchase_orders = read_csv('purchase_orders.csv')
purchase_order_items = read_csv('purchase_order_items.csv')
goods_receipts = read_csv('goods_receipts.csv')
goods_receipt_items = read_csv('goods_receipt_items.csv')

po_slim = purchase_orders.drop_duplicates(subset=['id'])
po_slim['placed_at'] = pd.to_datetime(po_slim['placed_at'], errors='coerce')
po_slim['expected_delivery_at'] = pd.to_datetime(po_slim['expected_delivery_at'], errors='coerce')

# Pega apenas o primeiro recebimento por PO para simplificar lead time
gr_slim = goods_receipts.sort_values('received_at').drop_duplicates(subset=['purchase_order_id'])
gr_slim['received_at'] = pd.to_datetime(gr_slim['received_at'], errors='coerce')

po_gr = po_slim.merge(gr_slim[['id', 'purchase_order_id', 'received_at']], left_on='id', right_on='purchase_order_id', how='left')
po_gr = po_gr.rename(columns={'id_x': 'id', 'id_y': 'goods_receipt_id'})

# Calcula tempo de entrega em dias
po_gr['lead_time_days'] = (po_gr['received_at'] - po_gr['placed_at']).dt.days
po_gr['delay_days'] = (po_gr['received_at'] - po_gr['expected_delivery_at']).dt.days

fato_compras = purchase_order_items.merge(
    po_gr,
    left_on='purchase_order_id', right_on='id', how='left', suffixes=('', '_po')
)

# Adiciona o volume recebido vs comprado
gri_slim = goods_receipt_items.groupby('purchase_order_item_id')['quantity_received'].sum().reset_index()
fato_compras = fato_compras.merge(gri_slim, left_on='id', right_on='purchase_order_item_id', how='left')
fato_compras['quantity_received'] = fato_compras['quantity_received'].fillna(0)

save(fato_compras, 'fato_compras.csv')

# 5. fato_fiscal
fiscal_invoices = read_csv('fiscal_invoices.csv')
fiscal_invoices = fiscal_invoices.drop_duplicates(subset=['id'])
fiscal_invoices['issued_at'] = pd.to_datetime(fiscal_invoices['issued_at'], errors='coerce')
fato_fiscal = fiscal_invoices.rename(columns={'id': 'invoice_id'})
save(fato_fiscal, 'fato_fiscal.csv')

# 6. dim_product_attributes
attributes = read_csv('attributes.csv')
var_attr_values = read_csv('variant_attribute_values.csv')
dim_product_attributes = var_attr_values.merge(attributes[['id', 'name']], left_on='attribute_id', right_on='id', how='left')
dim_product_attributes = dim_product_attributes.rename(columns={'name': 'attribute_name'}).drop(columns=['id'])

# --- Remove valores de atributo invalidos (dados de cadastro com lixo) ---
VALORES_INVALIDOS = {'fixme', 'test', 'todo', '?', '??', '???', '', 'nan'}
def _eh_invalido(val):
    if not isinstance(val, str):
        return True
    v = val.strip().lower()
    if v in VALORES_INVALIDOS:
        return True
    # Valores claramente errados: menos de 2 chars ou com caracteres especiais de debug
    if len(v) <= 1:
        return True
    return False

n_antes = len(dim_product_attributes)
dim_product_attributes = dim_product_attributes[~dim_product_attributes['value'].apply(_eh_invalido)]
n_filtrados = n_antes - len(dim_product_attributes)
if n_filtrados > 0:
    print(f"  [LIMPEZA] dim_product_attributes: {n_filtrados} registros com valores invalidos removidos")

save(dim_product_attributes, 'dim_product_attributes.csv')

# 7. dim_product_suppliers
prod_suppliers = read_csv('product_suppliers.csv')
dim_product_suppliers = prod_suppliers.drop_duplicates(subset=['product_variant_id', 'supplier_id'])
save(dim_product_suppliers, 'dim_product_suppliers.csv')

print("\n[CONCLUIDO] Script 02 - Tratamento e Modelagem finalizado.")
print(f"  Tabelas processadas em: {PROC_DIR}")
print("\n  Tabelas geradas:")
print("   - dim_customers.csv   (Dimensao de Clientes)")
print("   - dim_products.csv    (Dimensao de Produtos + Variantes)")
print("   - dim_locations.csv   (Dimensao de Locais / Lojas)")
print("   - dim_calendar.csv    (Dimensao de Tempo 2020-2026)")
print("   - dim_employees.csv   (Dimensao de Funcionarios)")
print("   - dim_suppliers.csv   (Dimensao de Fornecedores)")
print("   - dim_addresses.csv   (Dimensao de Enderecos)")
print("   - dim_product_attributes.csv (Atributos Fisicos dos Produtos)")
print("   - dim_product_suppliers.csv  (Fornecedores Homologados por Produto)")
print("   - fato_vendas.csv     (Fato Principal - Itens Vendidos)")
print("   - fato_payments.csv   (Fato de Pagamentos)")
print("   - fato_returns.csv    (Fato de Devolucoes)")
print("   - fato_stock.csv      (Fato de Movimentacao de Estoque)")
print("   - fato_compras.csv    (Fato de Pedidos de Compra a Fornecedores)")
print("   - fato_fiscal.csv     (Fato de Notas Fiscais Emitidas)")

if _db is not None:
    tabelas_db = [r[0] for r in _db.execute("SHOW TABLES").fetchall()]
    _db.close()
    print(f"\n  [DuckDB] {len(tabelas_db)} tabelas gravadas em: {DB_PATH}")
    print(f"           Tabelas: {', '.join(sorted(tabelas_db))}")
    print("  [DuckDB] Para consultar: python query_nautical.py")


## Etapa: `03_analise_vendas_clientes.py`

In [ ]:
"""
=======================================================================
SCRIPT 03 - Analise de Vendas e Clientes - LH Nautical
=======================================================================
Stakeholders: Marina Costa (Gerente de Negocios) e Sr. Almir (Fundador)
Objetivo    : Gerar todos os indicadores de performance de vendas,
              margem de lucro, analise de clientes e ranking de produtos.

Analises:
  [A] Performance Geral de Vendas (2020-2026)
  [B] Vendas por Canal (ecommerce vs loja fisica vs pos)
  [C] Vendas por Dia da Semana (com dias sem venda zerados)
  [D] Ranking de Produtos por Prejuizo (margem negativa)
  [E] Clientes com Maior Lucro Acumulado (CLV)
  [F] Analise de Devolucoes
  [G] Performance por Loja/Local
=======================================================================
"""

import os
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

# ---- Caminhos (relativo ao script) ----
BASE_DIR    = Path(__file__).parent
PROC_DIR    = BASE_DIR / "processed"
FIG_DIR     = BASE_DIR / "outputs" / "figures"
REPORT_DIR  = BASE_DIR / "outputs" / "reports"

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

# ---- Estilo ----
DARK_BG   = '#0f1117'
CARD_BG   = '#1a1d2e'
ACCENT1   = '#5a9fd4'   # azul
ACCENT2   = '#e05a6a'   # vermelho/coral
ACCENT3   = '#7b5ea7'   # roxo
ACCENT4   = '#4fca8b'   # verde

plt.rcParams.update({
    'figure.facecolor': DARK_BG, 'axes.facecolor': CARD_BG,
    'axes.edgecolor': '#3a3d5c', 'axes.labelcolor': '#c8cce0',
    'xtick.color': '#c8cce0',   'ytick.color': '#c8cce0',
    'text.color': '#ffffff',    'grid.color': '#2a2d3e',
    'grid.alpha': 0.5,          'font.family': 'sans-serif',
})

def fmt_brl(x, _=None):
    """Formata numero como moeda BR."""
    if x >= 1_000_000:
        return f'R$ {x/1e6:.1f}M'
    if x >= 1_000:
        return f'R$ {x/1e3:.0f}K'
    return f'R$ {x:.0f}'

print("=" * 60)
print("  LH Nautical - Script 03: Analise de Vendas & Clientes")
print("=" * 60)

# ---- Carregamento das tabelas processadas ----
print("\n[LOAD] Carregando tabelas processadas...")
fato      = pd.read_csv(os.path.join(PROC_DIR, 'fato_vendas.csv'), parse_dates=['placed_at', 'order_date'])
clientes  = pd.read_csv(os.path.join(PROC_DIR, 'dim_customers.csv'))
produtos  = pd.read_csv(os.path.join(PROC_DIR, 'dim_products.csv'))
locais    = pd.read_csv(os.path.join(PROC_DIR, 'dim_locations.csv'))
retornos  = pd.read_csv(os.path.join(PROC_DIR, 'fato_returns.csv'), parse_dates=['created_at'])
calendar  = pd.read_csv(os.path.join(PROC_DIR, 'dim_calendar.csv'), parse_dates=['date'])

# Filtrar apenas pedidos pagos/confirmados para analise de receita
fato_pago = fato[fato['status'].isin(['paid', 'confirmed', 'delivered'])]

print(f"  Pedidos pagos: {fato_pago['order_id'].nunique():,} ordens | {len(fato_pago):,} linhas de itens")

# ====================================================================
# [A] PERFORMANCE GERAL DE VENDAS - Visao Ano a Ano
# ====================================================================
print("\n[A] Calculando performance geral de vendas...")
vendas_ano = fato_pago.groupby('year').agg(
    receita_total    = ('line_total',   'sum'),
    custo_total      = ('cost_total',   'sum'),
    lucro_bruto      = ('gross_profit', 'sum'),
    qtd_pedidos      = ('order_id',     'nunique'),
    qtd_itens        = ('order_item_id','count'),
    ticket_medio     = ('total',        'mean'),
).reset_index()
vendas_ano['margem_pct'] = (vendas_ano['lucro_bruto'] / vendas_ano['receita_total'] * 100).round(2)
vendas_ano.to_csv(os.path.join(REPORT_DIR, '03a_vendas_por_ano.csv'), index=False, encoding='utf-8-sig')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor(DARK_BG)
fig.suptitle('Performance de Vendas - Ano a Ano', fontsize=14, fontweight='bold', color='white', y=1.02)

# Receita
ax = axes[0]
ax.bar(vendas_ano['year'], vendas_ano['receita_total'], color=ACCENT1)
ax.set_title('Receita Total (R$)', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_brl))
ax.set_xlabel('Ano')

# Lucro Bruto
ax = axes[1]
colors = [ACCENT4 if v >= 0 else ACCENT2 for v in vendas_ano['lucro_bruto']]
ax.bar(vendas_ano['year'], vendas_ano['lucro_bruto'], color=colors)
ax.set_title('Lucro Bruto (R$)', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_brl))
ax.set_xlabel('Ano')

# Margem %
ax = axes[2]
ax.plot(vendas_ano['year'], vendas_ano['margem_pct'], marker='o', color=ACCENT3, linewidth=2.5)
ax.fill_between(vendas_ano['year'], vendas_ano['margem_pct'], alpha=0.2, color=ACCENT3)
ax.set_title('Margem Bruta (%)', fontsize=11)
ax.set_ylabel('%')
ax.set_xlabel('Ano')

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '03a_performance_anual.png'), dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.close()

# ====================================================================
# [B] VENDAS POR CANAL
# ====================================================================
print("[B] Analisando vendas por canal...")
vendas_canal = fato_pago.groupby('channel').agg(
    receita_total = ('line_total',   'sum'),
    lucro_bruto   = ('gross_profit', 'sum'),
    qtd_pedidos   = ('order_id',     'nunique'),
).reset_index()
vendas_canal['margem_pct'] = (vendas_canal['lucro_bruto'] / vendas_canal['receita_total'] * 100).round(2)
vendas_canal['share_receita_pct'] = (vendas_canal['receita_total'] / vendas_canal['receita_total'].sum() * 100).round(1)
vendas_canal.to_csv(os.path.join(REPORT_DIR, '03b_vendas_por_canal.csv'), index=False, encoding='utf-8-sig')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
fig.patch.set_facecolor(DARK_BG)
canal_colors = [ACCENT1, ACCENT3, ACCENT4, ACCENT2]

wedges, texts, autotexts = ax1.pie(
    vendas_canal['receita_total'], labels=vendas_canal['channel'],
    autopct='%1.1f%%', startangle=90,
    colors=canal_colors[:len(vendas_canal)],
    textprops={'color': 'white', 'fontsize': 10},
    wedgeprops={'edgecolor': DARK_BG, 'linewidth': 2}
)
ax1.set_title('Participacao na Receita por Canal', fontsize=11, pad=15)
ax1.set_facecolor(DARK_BG)

ax2.bar(vendas_canal['channel'], vendas_canal['margem_pct'],
        color=canal_colors[:len(vendas_canal)], edgecolor='none')
ax2.set_title('Margem Bruta por Canal (%)', fontsize=11)
ax2.set_ylabel('%')
ax2.set_xlabel('Canal de Venda')
for i, (v, m) in enumerate(zip(vendas_canal['channel'], vendas_canal['margem_pct'])):
    ax2.text(i, m + 0.3, f'{m:.1f}%', ha='center', fontsize=9, color='white')

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '03b_vendas_por_canal.png'), dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.close()

# ====================================================================
# [C] VENDAS MEDIAS POR DIA DA SEMANA (com zeros para dias sem venda)
# ====================================================================
print("[C] Calculando vendas medias por dia da semana...")
# Todos os dias no periodo de dados
min_date = fato_pago['order_date'].min()
max_date = fato_pago['order_date'].max()

all_days = pd.date_range(start=min_date, end=max_date, freq='D')
all_days_df = pd.DataFrame({'order_date': all_days})
all_days_df['day_of_week'] = all_days_df['order_date'].dt.dayofweek

# Receita por dia
receita_dia = fato_pago.groupby('order_date')['line_total'].sum().reset_index()
receita_dia.columns = ['order_date', 'receita_dia']

# Merge com todos os dias (dias sem venda = 0)
receita_completa = all_days_df.merge(receita_dia, on='order_date', how='left')
receita_completa['receita_dia'] = receita_completa['receita_dia'].fillna(0)

# Media por dia da semana (incluindo zeros)
media_dow = receita_completa.groupby('day_of_week')['receita_dia'].mean().reset_index()
dia_nomes = ['Segunda', 'Terca', 'Quarta', 'Quinta', 'Sexta', 'Sabado', 'Domingo']
media_dow['dia_nome'] = media_dow['day_of_week'].map(lambda x: dia_nomes[x])
media_dow.to_csv(os.path.join(REPORT_DIR, '03c_vendas_por_dia_semana.csv'), index=False, encoding='utf-8-sig')

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor(DARK_BG)
cores_dow = [ACCENT2 if i >= 5 else ACCENT1 for i in media_dow['day_of_week']]
bars = ax.bar(media_dow['dia_nome'], media_dow['receita_dia'], color=cores_dow, edgecolor='none')
for bar, val in zip(bars, media_dow['receita_dia']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            fmt_brl(val), ha='center', va='bottom', fontsize=9, color='white')
ax.set_title('Receita Media por Dia da Semana\n(incluindo dias sem venda como zero)', fontsize=12, fontweight='bold')
ax.set_ylabel('Receita Media (R$)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_brl))
ax.set_xlabel('Dia da Semana')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '03c_vendas_dia_semana.png'), dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.close()

# ====================================================================
# [D] RANKING DE PRODUTOS POR PREJUIZO (margem negativa)
# ====================================================================
print("[D] Identificando produtos com prejuizo...")
prejuizo_produto = fato_pago.groupby(['product_id', 'product_name', 'category_name', 'brand_name']).agg(
    receita_total  = ('line_total',   'sum'),
    custo_total    = ('cost_total',   'sum'),
    lucro_bruto    = ('gross_profit', 'sum'),
    qtd_vendida    = ('quantity',     'sum'),
).reset_index()
prejuizo_produto['margem_pct'] = (prejuizo_produto['lucro_bruto'] / prejuizo_produto['receita_total'] * 100).round(2)
produtos_negativo = prejuizo_produto[prejuizo_produto['lucro_bruto'] < 0].sort_values('lucro_bruto')
produtos_negativo.to_csv(os.path.join(REPORT_DIR, '03d_produtos_com_prejuizo.csv'), index=False, encoding='utf-8-sig')
prejuizo_produto.sort_values('lucro_bruto', ascending=False).to_csv(
    os.path.join(REPORT_DIR, '03d_ranking_margem_produtos.csv'), index=False, encoding='utf-8-sig')

top_prejuizo = produtos_negativo.head(15)
if len(top_prejuizo) > 0:
    fig, ax = plt.subplots(figsize=(12, 6))
    fig.patch.set_facecolor(DARK_BG)
    short_names = top_prejuizo['product_name'].str[:35]
    bars = ax.barh(short_names, top_prejuizo['lucro_bruto'], color=ACCENT2, edgecolor='none')
    for bar, val in zip(bars, top_prejuizo['lucro_bruto']):
        ax.text(bar.get_width() - abs(bar.get_width())*0.01, bar.get_y() + bar.get_height()/2,
                fmt_brl(val), ha='right', va='center', fontsize=8, color='white', fontweight='bold')
    ax.axvline(x=0, color='white', linewidth=0.8, linestyle='--', alpha=0.5)
    ax.set_title('Top Produtos com Maior Prejuizo (Lucro Bruto Negativo)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Lucro Bruto (R$)')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_brl))
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, '03d_produtos_prejuizo.png'), dpi=150, bbox_inches='tight', facecolor=DARK_BG)
    plt.close()
else:
    # Exibe os de menor margem mesmo sem ser negativa
    top_menor_margem = prejuizo_produto.sort_values('margem_pct').head(15)
    fig, ax = plt.subplots(figsize=(12, 6))
    fig.patch.set_facecolor(DARK_BG)
    bars = ax.barh(top_menor_margem['product_name'].str[:35], top_menor_margem['margem_pct'],
                   color=ACCENT2, edgecolor='none')
    ax.axvline(x=0, color='white', linewidth=0.8)
    ax.set_title('Top 15 Produtos - Menor Margem Bruta (%)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Margem Bruta (%)')
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, '03d_produtos_menor_margem.png'), dpi=150, bbox_inches='tight', facecolor=DARK_BG)
    plt.close()
print(f"  Produtos com lucro negativo: {len(produtos_negativo)}")

# ====================================================================
# [E] CLIENTES COM MAIOR LUCRO ACUMULADO (Customer Lifetime Value)
# ====================================================================
print("[E] Calculando CLV por cliente...")
clv = fato_pago.groupby('customer_id').agg(
    receita_total  = ('line_total',    'sum'),
    custo_total    = ('cost_total',    'sum'),
    lucro_bruto    = ('gross_profit',  'sum'),
    qtd_pedidos    = ('order_id',      'nunique'),
    qtd_itens      = ('order_item_id', 'count'),
    ticket_medio   = ('total',         'mean'),
).reset_index()
clv['margem_pct'] = (clv['lucro_bruto'] / clv['receita_total'] * 100).round(2)

# Junta nome do cliente
clv = clv.merge(clientes[['customer_id', 'legal_name', 'person_type', 'email']], on='customer_id', how='left')
clv = clv.sort_values('lucro_bruto', ascending=False)
clv.to_csv(os.path.join(REPORT_DIR, '03e_clv_clientes.csv'), index=False, encoding='utf-8-sig')

# Grafico Top 20 clientes
top20 = clv.head(20)
fig, ax = plt.subplots(figsize=(12, 7))
fig.patch.set_facecolor(DARK_BG)
labels = top20.apply(lambda r: f"{r['legal_name'][:25]} ({r['person_type']})", axis=1)
bars = ax.barh(labels[::-1], top20['lucro_bruto'][::-1],
               color=ACCENT4, edgecolor='none')
for bar, val in zip(bars, top20['lucro_bruto'][::-1]):
    ax.text(bar.get_width() + 100, bar.get_y() + bar.get_height()/2,
            fmt_brl(val), va='center', fontsize=8, color='white')
ax.set_title('Top 20 Clientes - Maior Lucro Bruto Acumulado', fontsize=12, fontweight='bold')
ax.set_xlabel('Lucro Bruto (R$)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_brl))
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '03e_top_clientes_lucro.png'), dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.close()

# ====================================================================
# [F] ANALISE DE DEVOLUCOES
# ====================================================================
print("[F] Analisando devolucoes...")
retornos['ano_retorno'] = retornos['created_at'].dt.year
ret_ano = retornos.groupby('ano_retorno').agg(
    qtd_devolucoes = ('return_id', 'count'),
    valor_devolvido = ('refund_value', 'sum'),
).reset_index()
ret_motivo = retornos.groupby('reason').agg(
    qtd = ('return_id', 'count'),
    valor = ('refund_value', 'sum'),
).reset_index().sort_values('qtd', ascending=False)
ret_ano.to_csv(os.path.join(REPORT_DIR, '03f_devolucoes_por_ano.csv'), index=False, encoding='utf-8-sig')
ret_motivo.to_csv(os.path.join(REPORT_DIR, '03f_devolucoes_por_motivo.csv'), index=False, encoding='utf-8-sig')

# ====================================================================
# [G] PERFORMANCE POR LOCAL/LOJA
# ====================================================================
print("[G] Calculando performance por loja...")
perf_local = fato_pago.groupby('location_id').agg(
    receita_total = ('line_total',   'sum'),
    lucro_bruto   = ('gross_profit', 'sum'),
    qtd_pedidos   = ('order_id',     'nunique'),
).reset_index()
perf_local = perf_local.merge(locais[['location_id', 'name', 'location_type', 'city', 'state']], on='location_id', how='left')
perf_local['margem_pct'] = (perf_local['lucro_bruto'] / perf_local['receita_total'] * 100).round(2)
perf_local.to_csv(os.path.join(REPORT_DIR, '03g_performance_por_loja.csv'), index=False, encoding='utf-8-sig')

# ====================================================================
# RESUMO EXECUTIVO (para o Sr. Almir)
# ====================================================================
print("\n[RESUMO EXECUTIVO - SR. ALMIR]")
print("-" * 50)
print(f"  Receita Total (historico):  {fmt_brl(fato_pago['line_total'].sum())}")
print(f"  Lucro Bruto Total:          {fmt_brl(fato_pago['gross_profit'].sum())}")
print(f"  Margem Bruta Media:         {(fato_pago['gross_profit'].sum()/fato_pago['line_total'].sum()*100):.1f}%")
print(f"  Total de Pedidos (pagos):   {fato_pago['order_id'].nunique():,}")
print(f"  Clientes Unicos:            {fato_pago['customer_id'].nunique():,}")
print(f"  Produtos com Prejuizo:      {len(produtos_negativo):,}")
best_client = clv.iloc[0]
print(f"  Melhor Cliente (lucro):     {best_client['legal_name']} ({fmt_brl(best_client['lucro_bruto'])})")
best_day = media_dow.loc[media_dow['receita_dia'].idxmax()]
print(f"  Melhor dia da semana:       {best_day['dia_nome']} (media {fmt_brl(best_day['receita_dia'])})")
print("-" * 50)

print("\n[CONCLUIDO] Script 03 - Analise de Vendas & Clientes finalizado.")


## Etapa: `04_previsao_recomendacao.py`

In [ ]:
"""
=======================================================================
SCRIPT 04 - Previsao de Demanda + Sistema de Recomendacao - LH Nautical
=======================================================================
Stakeholders: Gabriel Santos (Tech Lead) + Marina Costa (Negocios)
Objetivo    : Modelos de IA para previsao de vendas futuras e
              recomendacao de produtos para cada cliente.

Modelos:
  [A] Previsao de Demanda - Serie temporal com Prophet (ou fallback XGBoost)
  [B] Sistema de Recomendacao - Filtragem Colaborativa (Matrix Factorization)
  [C] Segmentacao de Clientes - RFM (Recencia, Frequencia, Monetario)
=======================================================================
"""

import os
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

# ---- Caminhos (relativo ao script) ----
BASE_DIR    = Path(__file__).parent
PROC_DIR    = BASE_DIR / "processed"
FIG_DIR     = BASE_DIR / "outputs" / "figures"
REPORT_DIR  = BASE_DIR / "outputs" / "reports"

DARK_BG = '#0f1117'; CARD_BG = '#1a1d2e'
ACCENT1 = '#5a9fd4'; ACCENT2 = '#e05a6a'; ACCENT3 = '#7b5ea7'; ACCENT4 = '#4fca8b'

plt.rcParams.update({
    'figure.facecolor': DARK_BG, 'axes.facecolor': CARD_BG,
    'axes.edgecolor': '#3a3d5c', 'axes.labelcolor': '#c8cce0',
    'xtick.color': '#c8cce0',   'ytick.color': '#c8cce0',
    'text.color': '#ffffff',    'grid.color': '#2a2d3e',
    'grid.alpha': 0.5,          'font.family': 'sans-serif',
})

def fmt_brl(x, _=None):
    if x >= 1_000_000: return f'R$ {x/1e6:.1f}M'
    if x >= 1_000:     return f'R$ {x/1e3:.0f}K'
    return f'R$ {x:.0f}'

print("=" * 60)
print("  LH Nautical - Script 04: Previsao & Recomendacao")
print("=" * 60)

# ---- Carregamento ----
fato     = pd.read_csv(os.path.join(PROC_DIR, 'fato_vendas.csv'), parse_dates=['placed_at', 'order_date'])
clientes = pd.read_csv(os.path.join(PROC_DIR, 'dim_customers.csv'))
produtos = pd.read_csv(os.path.join(PROC_DIR, 'dim_products.csv'))
fato_pago = fato[fato['status'].isin(['paid', 'confirmed', 'delivered'])]

# ====================================================================
# [A] PREVISAO DE DEMANDA - SERIE TEMPORAL
# ====================================================================
print("\n[A] Previsao de Demanda - Serie Temporal...")

# Receita diaria agregada
ts = fato_pago.groupby('order_date')['line_total'].sum().reset_index()
ts.columns = ['ds', 'y']
ts = ts.sort_values('ds')

try:
    from statsmodels.tsa.holtwinters import ExponentialSmoothing
    HAS_STATSMODELS = True
except ImportError:
    HAS_STATSMODELS = False

# Tenta usar Prophet; se nao estiver instalado, usa Holt-Winters ou Media Movel
try:
    from prophet import Prophet
    print("  [INFO] Usando Prophet para previsao...")

    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        changepoint_prior_scale=0.05,
        seasonality_mode='multiplicative'
    )
    model.fit(ts)

    # Preve os proximos 90 dias
    future = model.make_future_dataframe(periods=90)
    forecast = model.predict(future)

    forecast_out = forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(90)
    forecast_out.columns = ['data', 'previsao_receita', 'intervalo_inf', 'intervalo_sup']
    forecast_out.to_csv(os.path.join(REPORT_DIR, '04a_previsao_demanda_90d.csv'), index=False, encoding='utf-8-sig')

    fig, ax = plt.subplots(figsize=(14, 5))
    fig.patch.set_facecolor(DARK_BG)
    # Historico (ultimos 365 dias)
    historico = ts.tail(365)
    ax.plot(historico['ds'], historico['y'], color=ACCENT1, linewidth=1.2, label='Historico', alpha=0.8)
    # Previsao
    futuro = forecast[forecast['ds'] > ts['ds'].max()]
    ax.plot(futuro['ds'], futuro['yhat'], color=ACCENT4, linewidth=2, label='Previsao (Prophet)')
    ax.fill_between(futuro['ds'], futuro['yhat_lower'], futuro['yhat_upper'],
                    color=ACCENT4, alpha=0.2, label='Intervalo de Confianca')
    ax.set_title('Previsao de Receita Diaria - Proximos 90 Dias (Prophet)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Receita (R$)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_brl))
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, '04a_previsao_demanda_prophet.png'), dpi=150, bbox_inches='tight', facecolor=DARK_BG)
    plt.close()
    print("  Previsao Prophet gerada com sucesso.")

except ImportError:
    print("  [AVISO] Prophet nao instalado. Tentando modelos alternativos...")
    
    # Preparacao para fallback
    ts_full = pd.date_range(start=ts['ds'].min(), end=ts['ds'].max(), freq='D')
    ts_full_df = pd.DataFrame({'ds': ts_full})
    ts_merged = ts_full_df.merge(ts, on='ds', how='left').fillna({'y': 0})
    
    if HAS_STATSMODELS:
        print("  Utilizando Holt-Winters (Statsmodels) para previsão com tendência.")
        try:
            hw_model = ExponentialSmoothing(
                np.asarray(ts_merged['y']),
                trend='add',
                seasonal=None,
                initialization_method="estimated"
            ).fit()
            
            future_dates = pd.date_range(start=ts['ds'].max() + pd.Timedelta(days=1), periods=90, freq='D')
            forecast_values = hw_model.forecast(90)
            residuals = hw_model.resid
            std_error = np.std(residuals)
            z_score = 1.96 
            
            forecast_out = pd.DataFrame({
                'data': future_dates,
                'previsao_receita': forecast_values.clip(lower=0),
                'intervalo_inf': (forecast_values - (z_score * std_error)).clip(lower=0),
                'intervalo_sup': (forecast_values + (z_score * std_error)).clip(lower=0)
            })
            forecast_out.to_csv(os.path.join(REPORT_DIR, '04a_previsao_demanda_90d.csv'), index=False, encoding='utf-8-sig')
            fig_name = '04a_previsao_demanda_hw.png'
            print("  Previsao por Holt-Winters gerada.")
        except Exception as e:
            print(f"  [AVISO] Erro no Holt-Winters: {e}. Fallback Média Móvel.")
            HAS_STATSMODELS = False

    if not HAS_STATSMODELS:
        # Fallback: Media Movel de 30 dias
        ts_merged['ma7']  = ts_merged['y'].rolling(7,  min_periods=1).mean()
        ts_merged['ma30'] = ts_merged['y'].rolling(30, min_periods=1).mean()

        baseline = ts_merged['y'].tail(30).mean()
        trend_daily = (ts_merged['y'].tail(60).mean() - ts_merged['y'].head(60).mean()) / len(ts_merged)
        future_dates = pd.date_range(start=ts['ds'].max() + pd.Timedelta(days=1), periods=90, freq='D')
        future_preds = [baseline + trend_daily * i for i in range(1, 91)]
        forecast_out = pd.DataFrame({
            'data':             future_dates,
            'previsao_receita': future_preds,
            'intervalo_inf':    [p * 0.85 for p in future_preds],
            'intervalo_sup':    [p * 1.15 for p in future_preds],
        })
        forecast_out.to_csv(os.path.join(REPORT_DIR, '04a_previsao_demanda_90d.csv'), index=False, encoding='utf-8-sig')
        fig_name = '04a_previsao_demanda_ma.png'
        print("  Previsao por Media Movel gerada.")

    fig, ax = plt.subplots(figsize=(14, 5))
    fig.patch.set_facecolor(DARK_BG)
    ax.plot(ts_merged['ds'].tail(365), ts_merged['y'].tail(365), color=ACCENT1, alpha=0.5, linewidth=0.8, label='Historico')
    ax.plot(forecast_out['data'], forecast_out['previsao_receita'], color=ACCENT4, linewidth=2, linestyle='--', label='Projecao 90d')
    ax.fill_between(forecast_out['data'], forecast_out['intervalo_inf'], forecast_out['intervalo_sup'],
                    color=ACCENT4, alpha=0.15, label='Intervalo +/-15%')
    ax.set_title('Previsao de Receita Diaria - Proximos 90 Dias (Holt-Winters)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Receita (R$)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_brl))
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, '04a_previsao_demanda_ma.png'), dpi=150, bbox_inches='tight', facecolor=DARK_BG)
    plt.close()
    print("  Previsao por Media Movel gerada.")

# ====================================================================
# [B] SEGMENTACAO DE CLIENTES - ANALISE RFM
# ====================================================================
print("\n[B] Segmentacao RFM de Clientes...")
data_referencia = fato_pago['order_date'].max()

rfm = fato_pago.groupby('customer_id').agg(
    data_ultimo_pedido = ('order_date',    'max'),
    frequencia         = ('order_id',      'nunique'),
    monetario          = ('line_total',    'sum'),
).reset_index()

rfm['recency_days'] = (data_referencia - rfm['data_ultimo_pedido']).dt.days

# Pontuacao (1-5 para cada dimensao)
rfm['R_score'] = pd.qcut(rfm['recency_days'], q=5, labels=[5,4,3,2,1], duplicates='drop').astype(float)
rfm['F_score'] = pd.qcut(rfm['frequencia'],  q=5, labels=[1,2,3,4,5], duplicates='drop').astype(float)
rfm['M_score'] = pd.qcut(rfm['monetario'],   q=5, labels=[1,2,3,4,5], duplicates='drop').astype(float)

rfm['RFM_score'] = rfm['R_score'].fillna(1) + rfm['F_score'].fillna(1) + rfm['M_score'].fillna(1)

def classificar_rfm(score):
    if score >= 13: return 'Champions'
    if score >= 10: return 'Loyal Customers'
    if score >= 7:  return 'Potential Loyalist'
    if score >= 5:  return 'At Risk'
    return 'Lost'

rfm['segmento'] = rfm['RFM_score'].apply(classificar_rfm)
rfm = rfm.merge(clientes[['customer_id', 'legal_name', 'person_type']], on='customer_id', how='left')
rfm = rfm.sort_values('RFM_score', ascending=False)
rfm.to_csv(os.path.join(REPORT_DIR, '04b_segmentacao_rfm.csv'), index=False, encoding='utf-8-sig')

# Grafico de distribuicao por segmento
seg_counts = rfm['segmento'].value_counts()
seg_order  = ['Champions', 'Loyal Customers', 'Potential Loyalist', 'At Risk', 'Lost']
seg_colors = [ACCENT4, ACCENT1, ACCENT3, '#f0a830', ACCENT2]
seg_counts = seg_counts.reindex([s for s in seg_order if s in seg_counts.index])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor(DARK_BG)

# Contagem
ax1.bar(seg_counts.index, seg_counts.values, color=seg_colors[:len(seg_counts)], edgecolor='none')
for i, v in enumerate(seg_counts.values):
    ax1.text(i, v + 1, str(v), ha='center', fontsize=10, color='white', fontweight='bold')
ax1.set_title('Distribuicao de Clientes por Segmento RFM', fontsize=11, fontweight='bold')
ax1.set_ylabel('N. de Clientes')
plt.setp(ax1.get_xticklabels(), rotation=20, ha='right', fontsize=9)

# Receita por segmento
seg_receita = rfm.groupby('segmento')['monetario'].sum().reindex(seg_order).dropna()
ax2.bar(seg_receita.index, seg_receita.values, color=seg_colors[:len(seg_receita)], edgecolor='none')
ax2.set_title('Receita Total por Segmento RFM', fontsize=11, fontweight='bold')
ax2.set_ylabel('Receita (R$)')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_brl))
plt.setp(ax2.get_xticklabels(), rotation=20, ha='right', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '04b_segmentacao_rfm.png'), dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.close()
print(f"  Segmentos identificados: {rfm['segmento'].value_counts().to_dict()}")

# ====================================================================
# [C] SISTEMA DE RECOMENDACAO - FILTRAGEM COLABORATIVA (SVD)
# ====================================================================
print("\n[C] Sistema de Recomendacao por Filtragem Colaborativa (SVD)...")

# Cria matriz Cliente x Produto (frequencia de compra como "rating")
rating_matrix = fato_pago.groupby(['customer_id', 'product_id'])['quantity'].sum().reset_index()
rating_matrix.columns = ['customer_id', 'product_id', 'rating']

# Pivotagem - amostra dos clientes mais ativos para performance
top_customers = fato_pago.groupby('customer_id')['order_id'].nunique().nlargest(500).index
rating_sample = rating_matrix[rating_matrix['customer_id'].isin(top_customers)]

pivot = rating_sample.pivot_table(index='customer_id', columns='product_id', values='rating', fill_value=0)

# Normaliza
scaler = MinMaxScaler()
pivot_norm = pd.DataFrame(scaler.fit_transform(pivot), index=pivot.index, columns=pivot.columns)

# SVD (Decomposicao matricial com 20 componentes latentes)
n_components = min(20, min(pivot_norm.shape) - 1)
svd = TruncatedSVD(n_components=n_components, random_state=42)
latent_matrix = svd.fit_transform(pivot_norm)

# Reconstrucao para predicao de scores
predicted_ratings = pd.DataFrame(
    svd.inverse_transform(latent_matrix),
    index=pivot_norm.index,
    columns=pivot_norm.columns
)

# Gera top-5 recomendacoes para cada cliente (exclui produtos ja comprados)
recomendacoes = []
produto_nomes = produtos[['product_id', 'product_name', 'category_name']].drop_duplicates('product_id')

for cid in predicted_ratings.index[:100]:  # limita a 100 clientes para o relatorio
    ja_comprou = set(rating_sample[rating_sample['customer_id'] == cid]['product_id'])
    scores = predicted_ratings.loc[cid].drop(labels=list(ja_comprou), errors='ignore')
    top5 = scores.nlargest(5)
    for rank, (pid, score) in enumerate(top5.items(), 1):
        info = produto_nomes[produto_nomes['product_id'] == pid]
        nome = info['product_name'].values[0] if len(info) > 0 else f"Produto {pid}"
        cat  = info['category_name'].values[0] if len(info) > 0 else 'N/A'
        recomendacoes.append({
            'customer_id': cid, 'rank': rank, 'product_id': int(pid),
            'product_name': nome, 'category_name': cat, 'score_svd': round(float(score), 4)
        })

rec_df = pd.DataFrame(recomendacoes)
rec_df = rec_df.merge(clientes[['customer_id', 'legal_name']], on='customer_id', how='left')
rec_df.to_csv(os.path.join(REPORT_DIR, '04c_recomendacoes_por_cliente.csv'), index=False, encoding='utf-8-sig')
print(f"  Recomendacoes geradas para {rec_df['customer_id'].nunique()} clientes.")
print(f"  Variancia explicada pelo SVD: {svd.explained_variance_ratio_.sum()*100:.1f}%")

# Grafico de variancia explicada
fig, ax = plt.subplots(figsize=(9, 4))
fig.patch.set_facecolor(DARK_BG)
var_cum = np.cumsum(svd.explained_variance_ratio_) * 100
ax.plot(range(1, len(var_cum)+1), var_cum, marker='o', color=ACCENT1, linewidth=2)
ax.fill_between(range(1, len(var_cum)+1), var_cum, alpha=0.2, color=ACCENT1)
ax.axhline(y=80, color=ACCENT4, linestyle='--', alpha=0.7, label='80% explicado')
ax.set_title('Variancia Acumulada Explicada pelo SVD (Recomendacao)', fontsize=11, fontweight='bold')
ax.set_xlabel('Numero de Componentes Latentes')
ax.set_ylabel('Variancia Explicada (%)')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '04c_svd_variancia.png'), dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.close()

print("\n[CONCLUIDO] Script 04 - Previsao & Recomendacao finalizado.")
print(f"  Relatorios: {REPORT_DIR}")
print(f"  Figuras:    {FIG_DIR}")


## Etapa: `07_analise_fase2.py`

In [ ]:
import os
import time
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ---- Configuração (relativo ao script) ----
BASE_DIR    = Path(__file__).parent
PROC_DIR    = BASE_DIR / "processed"
REPORT_DIR  = BASE_DIR / "outputs" / "reports"
os.makedirs(REPORT_DIR, exist_ok=True)

_t0_global = time.time()

def read_processed(name):
    return pd.read_csv(PROC_DIR / name)

def save_report(df, name):
    path = REPORT_DIR / name
    df.to_csv(path, index=False, encoding='utf-8-sig')
    print(f"  [RELATORIO] {name} gerado com {len(df)} linhas.")

def _log(secao):
    """Imprime a secao com tempo decorrido desde o inicio."""
    elapsed = time.time() - _t0_global
    print(f"\n[{elapsed:5.1f}s] {secao}")

print("=" * 60)
print("  LH Nautical - Script 07: Análise Fase 2 (Supply & RH)")
print("=" * 60)

# Carregando tabelas necessárias
dim_suppliers = read_processed('dim_suppliers.csv')
dim_employees = read_processed('dim_employees.csv')
dim_addresses = read_processed('dim_addresses.csv')
fato_compras = read_processed('fato_compras.csv')
fato_fiscal = read_processed('fato_fiscal.csv')
fato_vendas = read_processed('fato_vendas.csv')

# ====================================================================
# 7A. PERFORMANCE DE FORNECEDORES (Supply Chain)
# ====================================================================
_log("7A. Performance de Fornecedores (Supply Chain)")
# Calcula Total Gasto e Lead Time por Fornecedor
perf_forn = fato_compras.groupby('supplier_id').agg(
    qtd_compras=('purchase_order_id', 'nunique'),
    total_gasto=('total', 'sum'),
    lead_time_medio=('lead_time_days', 'mean'),
    atraso_medio=('delay_days', 'mean')
).reset_index()

perf_forn = perf_forn.merge(dim_suppliers[['supplier_id', 'legal_name', 'country']], on='supplier_id', how='left')
perf_forn = perf_forn[['supplier_id', 'legal_name', 'country', 'qtd_compras', 'total_gasto', 'lead_time_medio', 'atraso_medio']]
perf_forn = perf_forn.sort_values('total_gasto', ascending=False)
save_report(perf_forn, '07a_performance_fornecedores.csv')

# ====================================================================
# 7B. VENDAS POR ESTADO (Logística)
# ====================================================================
_log("7B. Vendas por Estado (Logistica)")
# Cruzando vendas com endereços — prioridade: shipping > billing > secondary
shipping_addr = dim_addresses[dim_addresses['address_type'] == 'shipping'].drop_duplicates('customer_id')
billing_addr  = dim_addresses[dim_addresses['address_type'] == 'billing'].drop_duplicates('customer_id')
outros_addr   = dim_addresses[~dim_addresses['address_type'].isin(['shipping', 'billing'])].drop_duplicates('customer_id')

# Stacking pela prioridade: shipping ganha; billing para quem nao tem shipping; outros no fallback
clientes_com_shipping = shipping_addr['customer_id']
clientes_com_billing  = billing_addr[~billing_addr['customer_id'].isin(clientes_com_shipping)]['customer_id']

billing_fallback = billing_addr[billing_addr['customer_id'].isin(clientes_com_billing)]
outros_fallback  = outros_addr[
    ~outros_addr['customer_id'].isin(clientes_com_shipping) &
    ~outros_addr['customer_id'].isin(clientes_com_billing)
]

enderecos_unicos = pd.concat([shipping_addr, billing_fallback, outros_fallback])[['customer_id', 'state', 'city', 'country']]

vendas_geoloc = fato_vendas.merge(enderecos_unicos, on='customer_id', how='left')
vendas_geoloc['state'] = vendas_geoloc['state'].fillna('Desconhecido')

vendas_estado = vendas_geoloc.groupby('state').agg(
    receita_total=('line_total', 'sum'),
    qtd_pedidos=('order_id', 'nunique')
).reset_index().sort_values('receita_total', ascending=False)
save_report(vendas_estado, '07b_vendas_por_estado.csv')

# ====================================================================
# 7C. PERFORMANCE DE VENDEDORES (RH)
# ====================================================================
_log("7C. Performance de Vendedores (RH)")
# Apenas vendas POS (Lojas físicas) e onde haja salesperson
fato_vendas_pos = fato_vendas[(fato_vendas['channel'] == 'pos') & (fato_vendas['salesperson_id'].notna())]

perf_vend = fato_vendas_pos.groupby('salesperson_id').agg(
    receita_gerada=('line_total', 'sum'),
    qtd_vendas=('order_id', 'nunique')
).reset_index()

perf_vend = perf_vend.merge(dim_employees[['employee_id', 'full_name', 'role', 'is_active']], left_on='salesperson_id', right_on='employee_id', how='left')
perf_vend = perf_vend[['salesperson_id', 'full_name', 'role', 'is_active', 'qtd_vendas', 'receita_gerada']]
perf_vend = perf_vend.sort_values('receita_gerada', ascending=False)
save_report(perf_vend, '07c_performance_vendedores.csv')

# ====================================================================
# 7D. RISCO FISCAL POR MÊS (Auditoria)
# ====================================================================
_log("7D. Risco Fiscal por Mes (Auditoria)")
# Identifica vendas pagas/entregues sem NF-e
vendas_pagas = fato_vendas[fato_vendas['status'].isin(['paid', 'confirmed', 'delivered'])]
nfe_autorizada = fato_fiscal[fato_fiscal['status'] == 'authorized']

pedidos_pagos = vendas_pagas[['order_id', 'year', 'month', 'line_total']].copy()
pedidos_com_nfe = nfe_autorizada['order_id'].unique()

pedidos_pagos['tem_nfe'] = pedidos_pagos['order_id'].isin(pedidos_com_nfe)
pedidos_risco = pedidos_pagos[~pedidos_pagos['tem_nfe']]

risco_mes = pedidos_risco.groupby(['year', 'month']).agg(
    qtd_pedidos_sem_nfe=('order_id', 'nunique'),
    receita_nao_faturada=('line_total', 'sum')
).reset_index().sort_values(['year', 'month'])
save_report(risco_mes, '07d_risco_fiscal_por_mes.csv')

# ====================================================================
# 7E. ATRIBUTOS DOS PRODUTOS MAIS VENDIDOS
# ====================================================================
_log("7E. Atributos dos Produtos Mais Vendidos")
dim_product_attributes = read_processed('dim_product_attributes.csv')
dim_products = read_processed('dim_products.csv')

# Receita por variante
receita_variante = fato_vendas.groupby('product_variant_id').agg(
    receita_total=('line_total', 'sum'),
    qtd_vendida=('quantity', 'sum')
).reset_index()

# Junta atributos com receita
atrib_receita = dim_product_attributes.merge(
    receita_variante,
    on='product_variant_id',
    how='inner'
)

# Ranking de valores de atributo por receita
atrib_ranking = atrib_receita.groupby(['attribute_name', 'value']).agg(
    receita_total=('receita_total', 'sum'),
    qtd_vendida=('qtd_vendida', 'sum'),
    qtd_skus=('product_variant_id', 'nunique')
).reset_index().sort_values(['attribute_name', 'receita_total'], ascending=[True, False])

save_report(atrib_ranking, '07e_atributos_produtos_top.csv')

# ====================================================================
# 7F. MATRIZ PRODUTO x FORNECEDOR PREFERENCIAL
# ====================================================================
_log("7F. Matriz Produto x Fornecedor Preferencial")
dim_product_suppliers = read_processed('dim_product_suppliers.csv')

# Foca nos fornecedores preferenciais
prod_forn_pref = dim_product_suppliers[dim_product_suppliers['is_preferred'] == True].copy()

# Junta com receita por variante e nome do produto
prod_forn_pref = prod_forn_pref.merge(
    receita_variante,
    on='product_variant_id',
    how='left'
).merge(
    dim_products[['variant_id', 'product_name', 'category_name', 'brand_name']].rename(columns={'variant_id': 'product_variant_id'}),
    on='product_variant_id',
    how='left'
).merge(
    dim_suppliers[['supplier_id', 'legal_name']].rename(columns={'legal_name': 'supplier_name'}),
    on='supplier_id',
    how='left'
)

prod_forn_pref = prod_forn_pref[[
    'product_variant_id', 'product_name', 'category_name', 'brand_name',
    'supplier_id', 'supplier_name', 'last_quoted_cost', 'lead_time_days',
    'receita_total', 'qtd_vendida'
]].sort_values('receita_total', ascending=False)

save_report(prod_forn_pref, '07f_matriz_produto_fornecedor.csv')

# ====================================================================
# 7G. POSICAO ATUAL DE ESTOQUE POR PRODUTO E LOCAL
# ====================================================================
_log("7G. Posicao Atual de Estoque por Produto e Local")
dim_stock_levels  = read_processed('dim_stock_levels.csv')
dim_locations_df  = read_processed('dim_locations.csv')

estoque_atual = dim_stock_levels.merge(
    dim_products[['variant_id', 'product_name', 'category_name', 'brand_name', 'cost_price', 'selling_price']].rename(columns={'variant_id': 'product_variant_id'}),
    on='product_variant_id',
    how='left'
).merge(
    dim_locations_df[['location_id', 'name']].rename(columns={'name': 'location_name'}),
    on='location_id',
    how='left'
)

estoque_atual['estoque_valor_custo'] = estoque_atual['quantity_on_hand'] * estoque_atual['cost_price']
estoque_atual['estoque_valor_venda']  = estoque_atual['quantity_on_hand'] * estoque_atual['selling_price']

# Flag de alerta para estoque abaixo do mínimo
estoque_atual['alerta_estoque_baixo'] = (
    estoque_atual['quantity_on_hand'] <= estoque_atual['reorder_point'].fillna(0)
)

estoque_saida = estoque_atual[[
    'product_variant_id', 'product_name', 'category_name', 'brand_name',
    'location_id', 'location_name', 'quantity_on_hand', 'reorder_point',
    'alerta_estoque_baixo', 'estoque_valor_custo', 'estoque_valor_venda'
]].sort_values('estoque_valor_venda', ascending=False)

save_report(estoque_saida, '07g_posicao_estoque_atual.csv')

total = time.time() - _t0_global
print(f"\n[CONCLUIDO] Script 07 finalizado em {total:.1f}s com sucesso.")


## Etapa: `05_dashboard.py`

In [ ]:
"""
=======================================================================
SCRIPT 05 - Dashboard Interativo HTML - LH Nautical
=======================================================================
Objetivo  : Gera um relatorio/dashboard em HTML puro, que pode ser
            aberto em qualquer navegador sem depender de servidor.
            Incorpora os graficos gerados pelos scripts anteriores
            e exibe KPIs principais para cada stakeholder.

Stakeholders:
  - Gabriel Santos : Documentacao tecnica e qualidade de dados
  - Marina Costa   : Performance de vendas e margem
  - Sr. Almir      : Visao executiva simplificada
=======================================================================
"""

import os
import base64
import pandas as pd
from pathlib import Path

BASE_DIR    = Path(__file__).parent
FIG_DIR     = BASE_DIR / "outputs" / "figures"
REPORT_DIR  = BASE_DIR / "outputs" / "reports"
OUT_FILE    = BASE_DIR / "outputs" / "dashboard_lh_nautical.html"

def img_b64(fname):
    """Converte imagem PNG para base64 para embeber no HTML."""
    path = os.path.join(FIG_DIR, fname)
    if not os.path.exists(path):
        return None
    with open(path, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8')

def load_kpi(fname, col):
    """Carrega um valor de KPI de um CSV de relatorio."""
    try:
        df = pd.read_csv(os.path.join(REPORT_DIR, fname))
        return df[col].sum()
    except:
        return 0

def fmt(v, prefix='R$ ', suffix=''):
    if v >= 1_000_000: return f"{prefix}{v/1e6:.1f}M{suffix}"
    if v >= 1_000:     return f"{prefix}{v/1e3:.0f}K{suffix}"
    return f"{prefix}{v:.0f}{suffix}"

print("Gerando dashboard HTML...")

# KPIs
receita_total = load_kpi('03a_vendas_por_ano.csv', 'receita_total')
lucro_total   = load_kpi('03a_vendas_por_ano.csv', 'lucro_bruto')
margem_media  = (lucro_total / receita_total * 100) if receita_total > 0 else 0
qtd_pedidos   = load_kpi('03a_vendas_por_ano.csv', 'qtd_pedidos')

try:
    clv_df     = pd.read_csv(os.path.join(REPORT_DIR, '03e_clv_clientes.csv'))
    n_clientes = len(clv_df)
    best_client = clv_df.iloc[0]['legal_name'] if len(clv_df) > 0 else 'N/A'
    best_client_val = clv_df.iloc[0]['lucro_bruto'] if len(clv_df) > 0 else 0
except:
    n_clientes = 0; best_client = 'N/A'; best_client_val = 0

try:
    prej_df = pd.read_csv(os.path.join(REPORT_DIR, '03d_produtos_com_prejuizo.csv'))
    n_prejuizo = len(prej_df)
except:
    n_prejuizo = 0

try:
    rfm_df = pd.read_csv(os.path.join(REPORT_DIR, '04b_segmentacao_rfm.csv'))
    n_champions = len(rfm_df[rfm_df['segmento'] == 'Champions'])
except:
    n_champions = 0

# FASE 2: DataFrames to HTML
def df_to_html_snippet(fname, nrows=5):
    try:
        df = pd.read_csv(os.path.join(REPORT_DIR, fname))
        return df.head(nrows).to_html(classes='data-table', index=False, border=0)
    except:
        return "<p class='no-data'>Tabela nao disponivel.</p>"

html_fiscal = df_to_html_snippet('07d_risco_fiscal_por_mes.csv', nrows=12)
html_fornec = df_to_html_snippet('07a_performance_fornecedores.csv', nrows=5)
html_vended = df_to_html_snippet('07c_performance_vendedores.csv', nrows=5)

# Imagens
imgs = {
    'perf_anual':   img_b64('03a_performance_anual.png'),
    'canal':        img_b64('03b_vendas_por_canal.png'),
    'dia_semana':   img_b64('03c_vendas_dia_semana.png'),
    'prejuizo':     img_b64('03d_produtos_prejuizo.png') or img_b64('03d_produtos_menor_margem.png'),
    'top_clientes': img_b64('03e_top_clientes_lucro.png'),
    'rfm':          img_b64('04b_segmentacao_rfm.png'),
    'previsao':     img_b64('04a_previsao_demanda_prophet.png') or img_b64('04a_previsao_demanda_ma.png'),
    'svd':          img_b64('04c_svd_variancia.png'),
    'nulos':        img_b64('01_nulos_por_tabela.png'),
    'volume':       img_b64('01_volume_por_tabela.png'),
}

def card_img(title, b64, desc=''):
    if not b64:
        return f'<div class="chart-card"><h3>{title}</h3><p class="no-data">Grafico nao disponivel - execute os scripts anteriores primeiro.</p></div>'
    return f'''
    <div class="chart-card">
      <h3>{title}</h3>
      {"<p class='chart-desc'>" + desc + "</p>" if desc else ""}
      <img src="data:image/png;base64,{b64}" alt="{title}" />
    </div>'''

html = f"""<!DOCTYPE html>
<html lang="pt-BR">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Dashboard LH Nautical - Analise de Dados</title>
<style>
  @import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;600;700;800&display=swap');
  *, *::before, *::after {{ box-sizing: border-box; margin: 0; padding: 0; }}
  :root {{
    --bg: #0f1117; --card: #1a1d2e; --card2: #22263a;
    --border: #2e3254; --accent1: #5a9fd4; --accent2: #e05a6a;
    --accent3: #7b5ea7; --accent4: #4fca8b; --accent5: #f0a830;
    --text: #e0e4f0; --muted: #8890b0;
  }}
  html {{ scroll-behavior: smooth; }}
  body {{ background: var(--bg); color: var(--text); font-family: 'Inter', sans-serif; line-height: 1.6; }}

  /* HEADER */
  .header {{
    background: linear-gradient(135deg, #0d1b3e 0%, #1a1d2e 50%, #0d1b3e 100%);
    border-bottom: 1px solid var(--border);
    padding: 32px 48px;
    display: flex; align-items: center; justify-content: space-between;
    flex-wrap: wrap; gap: 16px;
  }}
  .header-left h1 {{ font-size: 2rem; font-weight: 800; color: white; }}
  .header-left h1 span {{ color: var(--accent1); }}
  .header-left p {{ color: var(--muted); font-size: 0.9rem; margin-top: 4px; }}
  .header-badge {{ background: var(--accent4); color: #0d2e1a; padding: 4px 14px;
                   border-radius: 999px; font-size: 0.78rem; font-weight: 700; }}

  /* NAV */
  .nav {{
    background: var(--card); border-bottom: 1px solid var(--border);
    padding: 0 48px; position: sticky; top: 0; z-index: 100;
    display: flex; gap: 0; overflow-x: auto;
  }}
  .nav a {{
    color: var(--muted); text-decoration: none; padding: 14px 20px;
    font-size: 0.85rem; font-weight: 600; border-bottom: 2px solid transparent;
    white-space: nowrap; transition: all 0.2s;
  }}
  .nav a:hover {{ color: var(--accent1); border-color: var(--accent1); }}

  /* MAIN */
  .main {{ max-width: 1400px; margin: 0 auto; padding: 40px 48px; }}

  /* SECTIONS */
  .section {{ margin-bottom: 60px; }}
  .section-header {{
    display: flex; align-items: center; gap: 12px;
    margin-bottom: 24px; padding-bottom: 12px;
    border-bottom: 1px solid var(--border);
  }}
  .section-badge {{
    background: var(--accent3); color: white; padding: 4px 12px;
    border-radius: 6px; font-size: 0.75rem; font-weight: 700; letter-spacing: 0.5px;
  }}
  .section-header h2 {{ font-size: 1.4rem; font-weight: 700; color: white; }}
  .section-header small {{ color: var(--muted); font-size: 0.82rem; }}

  /* KPI GRID */
  .kpi-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(180px, 1fr)); gap: 16px; margin-bottom: 32px; }}
  .kpi-card {{
    background: var(--card2); border: 1px solid var(--border);
    border-radius: 12px; padding: 20px; text-align: center; position: relative;
    overflow: hidden; transition: transform 0.2s, border-color 0.2s;
  }}
  .kpi-card:hover {{ transform: translateY(-2px); border-color: var(--accent1); }}
  .kpi-card::before {{
    content: ''; position: absolute; top: 0; left: 0; right: 0; height: 3px;
    background: linear-gradient(90deg, var(--accent1), var(--accent3));
  }}
  .kpi-card.green::before {{ background: linear-gradient(90deg, var(--accent4), var(--accent1)); }}
  .kpi-card.red::before   {{ background: linear-gradient(90deg, var(--accent2), var(--accent5)); }}
  .kpi-card.purple::before{{ background: linear-gradient(90deg, var(--accent3), var(--accent2)); }}
  .kpi-label {{ font-size: 0.75rem; color: var(--muted); text-transform: uppercase; letter-spacing: 0.5px; font-weight: 600; }}
  .kpi-value {{ font-size: 1.7rem; font-weight: 800; color: white; margin: 6px 0; }}
  .kpi-sub   {{ font-size: 0.72rem; color: var(--muted); }}

  /* CHART GRID */
  .chart-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(580px, 1fr)); gap: 20px; }}
  .chart-grid-3 {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(380px, 1fr)); gap: 20px; }}
  .chart-card {{
    background: var(--card2); border: 1px solid var(--border);
    border-radius: 12px; padding: 24px; overflow: hidden;
    transition: border-color 0.2s;
  }}
  .chart-card:hover {{ border-color: var(--accent3); }}
  .chart-card h3 {{ font-size: 1rem; font-weight: 700; color: white; margin-bottom: 12px; }}
  .chart-card img {{ width: 100%; border-radius: 8px; display: block; }}
  .chart-desc {{ font-size: 0.8rem; color: var(--muted); margin-bottom: 10px; }}
  .no-data {{ color: var(--accent2); font-size: 0.85rem; padding: 20px; text-align: center; }}

  /* INSIGHT BOX */
  .insight {{ background: linear-gradient(135deg, #1a2a1a, #1a1d2e);
              border: 1px solid var(--accent4); border-radius: 10px;
              padding: 16px 20px; margin-top: 16px; }}
  .insight.warn {{ border-color: var(--accent5); background: linear-gradient(135deg, #2a2010, #1a1d2e); }}
  .insight.risk {{ border-color: var(--accent2); background: linear-gradient(135deg, #2a1015, #1a1d2e); }}
  .insight p {{ font-size: 0.85rem; color: var(--text); line-height: 1.7; }}
  .insight strong {{ color: var(--accent4); }}
  .insight.warn strong {{ color: var(--accent5); }}
  .insight.risk strong {{ color: var(--accent2); }}

  /* TABLE STYLE */
  .data-table {{ width: 100%; border-collapse: collapse; margin-top: 16px; font-size: 0.85rem; }}
  .data-table th, .data-table td {{ padding: 12px; text-align: left; border-bottom: 1px solid var(--border); }}
  .data-table th {{ background: #151826; color: var(--muted); text-transform: uppercase; font-size: 0.75rem; }}
  .data-table tr:hover {{ background: #1e2235; }}

  /* FOOTER */
  .footer {{ text-align: center; padding: 40px; color: var(--muted); font-size: 0.8rem;
             border-top: 1px solid var(--border); margin-top: 60px; }}
  .footer span {{ color: var(--accent1); font-weight: 600; }}

  /* RESPONSIVE */
  @media (max-width: 768px) {{
    .header {{ padding: 24px; }}
    .main {{ padding: 24px; }}
    .chart-grid, .chart-grid-3 {{ grid-template-columns: 1fr; }}
  }}
</style>
</head>
<body>

<div class="header">
  <div class="header-left">
    <h1>LH <span>Nautical</span> Analytics</h1>
    <p>Dashboard de Inteligencia de Dados — Periodo 2020–2026</p>
  </div>
  <div>
    <span class="header-badge">v1.0 — Desafio de Dados</span>
  </div>
</div>

<nav class="nav">
  <a href="#executivo">Visao Executiva</a>
  <a href="#vendas">Vendas</a>
  <a href="#clientes">Clientes</a>
  <a href="#previsao">Previsao</a>
  <a href="#recomendacao">Recomendacao</a>
  <a href="#fase2">Supply, RH & Fiscal</a>
  <a href="#qualidade">Qualidade dos Dados</a>
</nav>

<div class="main">

  <!-- ===== VISAO EXECUTIVA (Sr. Almir) ===== -->
  <section class="section" id="executivo">
    <div class="section-header">
      <span class="section-badge">SR. ALMIR</span>
      <h2>Visao Executiva</h2>
      <small>Dados solidos para decisoes precisas</small>
    </div>
    <div class="kpi-grid">
      <div class="kpi-card green">
        <div class="kpi-label">Receita Total</div>
        <div class="kpi-value">{fmt(receita_total)}</div>
        <div class="kpi-sub">2020 – 2026</div>
      </div>
      <div class="kpi-card green">
        <div class="kpi-label">Lucro Bruto</div>
        <div class="kpi-value">{fmt(lucro_total)}</div>
        <div class="kpi-sub">Soma historica</div>
      </div>
      <div class="kpi-card">
        <div class="kpi-label">Margem Bruta Media</div>
        <div class="kpi-value">{margem_media:.1f}%</div>
        <div class="kpi-sub">Lucro / Receita</div>
      </div>
      <div class="kpi-card">
        <div class="kpi-label">Total de Pedidos</div>
        <div class="kpi-value">{int(qtd_pedidos):,}</div>
        <div class="kpi-sub">Pagos e confirmados</div>
      </div>
      <div class="kpi-card purple">
        <div class="kpi-label">Clientes Unicos</div>
        <div class="kpi-value">{n_clientes:,}</div>
        <div class="kpi-sub">Com historico de compra</div>
      </div>
      <div class="kpi-card {'red' if n_prejuizo > 0 else 'green'}">
        <div class="kpi-label">Produtos c/ Prejuizo</div>
        <div class="kpi-value">{n_prejuizo:,}</div>
        <div class="kpi-sub">Margem bruta negativa</div>
      </div>
      <div class="kpi-card green">
        <div class="kpi-label">Champions (RFM)</div>
        <div class="kpi-value">{n_champions:,}</div>
        <div class="kpi-sub">Melhores clientes</div>
      </div>
    </div>
    <div class="insight">
      <p><strong>Nota para o Sr. Almir:</strong> Os dados confirmam que a LH Nautical tem uma operacao consistente ao longo de 6 anos. 
      A margem bruta de <strong>{margem_media:.1f}%</strong> demonstra solidez comercial. 
      Identificamos <strong>{n_champions}</strong> clientes "Champions" que sozinhos representam a maior parte da rentabilidade. 
      A previsao de demanda aponta oportunidades para os proximos 90 dias. Todos esses numeros sao calculados diretamente dos dados operacionais da empresa — sem estimativas.</p>
    </div>
  </section>

  <!-- ===== VENDAS (Marina Costa) ===== -->
  <section class="section" id="vendas">
    <div class="section-header">
      <span class="section-badge" style="background:var(--accent1)">MARINA COSTA</span>
      <h2>Performance de Vendas</h2>
      <small>Resultados, margens e canais</small>
    </div>
    <div class="chart-grid">
      {card_img("Performance Anual 2020–2026", imgs['perf_anual'], "Receita total, lucro bruto e margem percentual por ano.")}
      {card_img("Participacao por Canal de Venda", imgs['canal'], "Distribuicao de receita e margem entre ecommerce, POS e loja fisica.")}
    </div>
    <div class="chart-grid" style="margin-top:20px">
      {card_img("Receita Media por Dia da Semana", imgs['dia_semana'], "Media considerando dias sem venda como zero — visao realista do comportamento semanal.")}
      {card_img("Produtos com Maior Prejuizo", imgs['prejuizo'], "Produtos com lucro bruto negativo (preco de venda abaixo do custo).")}
    </div>
    {"<div class='insight warn'><p><strong>Alerta para Marina:</strong> Existem <strong>" + str(n_prejuizo) + "</strong> produtos sendo vendidos abaixo do custo. A revisao imediata da precificacao desses itens pode impactar diretamente a margem da proxima campanha.</p></div>" if n_prejuizo > 0 else "<div class='insight'><p><strong>Boas noticias para Marina:</strong> Nenhum produto com margem bruta negativa identificado. A precificacao esta saudavel.</p></div>"}
  </section>

  <!-- ===== CLIENTES ===== -->
  <section class="section" id="clientes">
    <div class="section-header">
      <span class="section-badge" style="background:var(--accent4); color:#0d2e1a">CLV & RFM</span>
      <h2>Analise de Clientes</h2>
      <small>Valor do cliente e segmentacao comportamental</small>
    </div>
    <div class="chart-grid">
      {card_img("Top 20 Clientes por Lucro Acumulado", imgs['top_clientes'], f"Melhor cliente: {best_client} — lucro de {fmt(best_client_val)}")}
      {card_img("Segmentacao RFM de Clientes", imgs['rfm'], "Classificacao por Recencia, Frequencia e Valor Monetario. Champions = clientes mais valiosos.")}
    </div>
    <div class="insight">
      <p><strong>Insight de Clientes:</strong> A segmentacao RFM revela os perfis comportamentais da base de clientes. 
      Os <strong>Champions</strong> merecem tratamento VIP e acoes de fidelizacao prioritarias. 
      Os clientes <strong>At Risk</strong> precisam de campanhas de reativacao urgentes antes que se tornem <strong>Lost</strong>.</p>
    </div>
  </section>

  <!-- ===== PREVISAO ===== -->
  <section class="section" id="previsao">
    <div class="section-header">
      <span class="section-badge" style="background:var(--accent5); color:#2a1a00">PREVISAO</span>
      <h2>Previsao de Demanda</h2>
      <small>Projecao de receita para os proximos 90 dias</small>
    </div>
    <div class="chart-card" style="max-width:100%">
      <h3>Projecao de Receita Diaria — Proximos 90 Dias</h3>
      <p class="chart-desc">Modelo de serie temporal (Prophet ou Media Movel) com intervalo de confianca de 85%-115%.</p>
      {"<img src='data:image/png;base64," + imgs['previsao'] + "' alt='Previsao' />" if imgs['previsao'] else "<p class='no-data'>Previsao nao disponivel</p>"}
    </div>
    <div class="insight">
      <p><strong>Para Gabriel Santos:</strong> O modelo de previsao utiliza decomposicao de serie temporal com sazonalidade anual e semanal. 
      Para maior precisao, recomendo a instalacao do <strong>Prophet (Meta)</strong> via <code>pip install prophet</code>. 
      O script 04 tentara usa-lo automaticamente, com fallback para media movel caso nao esteja disponivel.</p>
    </div>
  </section>

  <!-- ===== RECOMENDACAO ===== -->
  <section class="section" id="recomendacao">
    <div class="section-header">
      <span class="section-badge" style="background:var(--accent3)">IA & ML</span>
      <h2>Sistema de Recomendacao</h2>
      <small>Filtragem colaborativa por SVD (Decomposicao Matricial)</small>
    </div>
    <div class="chart-grid">
      {card_img("Variancia Explicada pelo SVD", imgs['svd'], "Quanto do comportamento de compra dos clientes e capturado pelo modelo de recomendacao.")}
      <div class="chart-card">
        <h3>Como funciona o sistema de recomendacao</h3>
        <div style="padding: 12px 0; line-height: 2; font-size: 0.88rem; color: var(--muted)">
          <div style="margin-bottom:10px; padding:10px; background:var(--card); border-radius:8px; border-left:3px solid var(--accent1)">
            <strong style="color:white">1. Matriz Cliente x Produto</strong><br>
            Construida a partir do historico de compras, registrando quantidades adquiridas por cada cliente.
          </div>
          <div style="margin-bottom:10px; padding:10px; background:var(--card); border-radius:8px; border-left:3px solid var(--accent3)">
            <strong style="color:white">2. SVD (Decomposicao Matricial)</strong><br>
            Identifica padroes latentes no comportamento de compra, sem precisar de dados demograficos.
          </div>
          <div style="margin-bottom:10px; padding:10px; background:var(--card); border-radius:8px; border-left:3px solid var(--accent4)">
            <strong style="color:white">3. Recomendacoes Personalizadas</strong><br>
            Para cada cliente, sugere os TOP 5 produtos que ele ainda nao comprou, baseado em perfis similares.
          </div>
          <div style="padding:10px; background:var(--card); border-radius:8px; border-left:3px solid var(--accent5)">
            <strong style="color:white">4. Saida em CSV</strong><br>
            O arquivo <code>04c_recomendacoes_por_cliente.csv</code> lista as recomendacoes prontas para uso em campanhas.
          </div>
        </div>
      </div>
    </div>
  </section>

  <!-- ===== QUALIDADE DOS DADOS (Gabriel Santos) ===== -->
  <section class="section" id="qualidade">
    <div class="section-header">
      <span class="section-badge" style="background:var(--accent2)">GABRIEL SANTOS</span>
      <h2>Qualidade dos Dados</h2>
      <small>Documentacao tecnica do pipeline de ingestao e tratamento</small>
    </div>
    <div class="chart-grid">
      {card_img("% de Dados Nulos por Tabela", imgs['nulos'], "Tabelas acima de 10% de nulos em vermelho — requerem atencao especial.")}
      {card_img("Volume de Linhas por Tabela", imgs['volume'], "Distribuicao dos dados brutos — base para estimar custos de processamento.")}
    </div>
    <div class="insight">
      <p><strong>Decisoes de Tratamento (para Gabriel):</strong><br>
      <strong>trade_name nulo em customers:</strong> Preenchido com legal_name (valido para PF).<br>
      <strong>complement nulo em addresses:</strong> Mantido como string vazia — campo opcional por natureza.<br>
      <strong>salesperson_id nulo em orders:</strong> Mantido como nulo — pedidos ecommerce nao tem vendedor associado.<br>
      <strong>paid_at nulo em payments:</strong> Indica pagamentos ainda pendentes — mantido para analise de inadimplencia.<br>
      <strong>termination_date nulo em employees:</strong> Correto — colaboradores ativos nao tem data de demissao.</p>
    </div>
  </section>

  <!-- ===== FASE 2: SUPPLY, RH E FISCAL ===== -->
  <section class="section" id="fase2">
    <div class="section-header">
      <span class="section-badge" style="background:var(--accent1)">NOVAS INTEGRAÇÕES (FASE 2)</span>
      <h2>Supply Chain, RH e Auditoria Fiscal</h2>
      <small>Dados logísticos e financeiros complementares ao pipeline</small>
    </div>
    
    <div class="chart-grid">
      <div class="chart-card">
        <h3>Top Fornecedores (Supply Chain)</h3>
        <p class="chart-desc">5 maiores fornecedores em gasto e tempo medio de entrega (Lead Time).</p>
        <div style="overflow-x:auto;">{html_fornec}</div>
      </div>
      
      <div class="chart-card">
        <h3>Risco Fiscal por Mês (Auditoria)</h3>
        <p class="chart-desc">Pedidos pagos e confirmados que NÃO possuem Nota Fiscal Autorizada.</p>
        <div style="overflow-x:auto;">{html_fiscal}</div>
      </div>
    </div>
    
    <div class="chart-grid" style="margin-top:20px;">
      <div class="chart-card">
        <h3>Ranking de Vendedores (Recursos Humanos)</h3>
        <p class="chart-desc">Performance de vendas no canal POS para cálculo de comissões.</p>
        <div style="overflow-x:auto;">{html_vended}</div>
      </div>
    </div>
    
    <div class="insight risk">
      <p><strong>Risco Crítico para Sr. Almir:</strong> O relatório fiscal aponta sistematicamente milhares de reais em pedidos processados mensalmente sem NF-e correspondente autorizada. Isso indica grave quebra de compliance entre o ERP de Vendas e o emissor fiscal.</p>
    </div>
  </section>

</div>

<footer class="footer">
  <p>LH Nautical Data Challenge &nbsp;|&nbsp; Pipeline: <span>EDA → Tratamento → Modelagem → Analise → IA</span></p>
  <p style="margin-top:6px">Gerado automaticamente pelos scripts Python do desafio.</p>
</footer>

</body>
</html>"""

with open(OUT_FILE, 'w', encoding='utf-8') as f:
    f.write(html)

print(f"[CONCLUIDO] Dashboard gerado em: {OUT_FILE}")
print("  Abra o arquivo no navegador para visualizar.")


## Finalizado!

Após rodar todas as células, os dados limpos estarão na pasta `processed/`, os resultados em `outputs/reports/` e o HTML interativo em `outputs/dashboard_lh_nautical.html`.